<a href="https://colab.research.google.com/github/mcundins/cse6242-project/blob/KMeansModel/kmeans_modeling/kmeans_food_desert_risk_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# K-means clustering (unsupervised learning)

Testing model with 2025 dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, confusion_matrix
import pickle

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
with open("/content/drive/MyDrive/Food_Desert_Data/cluster_model.pkl", "rb") as f:
    bundle = pickle.load(f)
pipe, model_cols, id_col = bundle["pipeline"], bundle["model_cols"], bundle["id_col"]

df_2025 = pd.read_csv("/content/drive/MyDrive/Food_Desert_Data/model_input_2025.csv", dtype={id_col: str})


In [4]:
df_2025.head()

,tract_id,clean__Urban,clean__population_total,clean__occupied_housing_units_total,clean__GroupQuartersFlag,clean__NUMGQTRS,clean__PCTGQTRS,clean__TractKids,clean__TractSeniors,clean__TractWhite,...,clean__PovertyRate,income_with_endpoint_proxy_2018_usd,clean__TractLOWI,clean__TractHUNV,clean__TractSNAP,clean__LATracts_half,clean__LATracts1,clean__LATractsVehicle_20,clean__low_access_population_1_20,FD_2
0,01001020100,1.0,1775.0,646.0,0.0,0.0,0.00,405.0,284.0,1389.0,...,17.4,61431.919806,1316.0,35.0,100.0,1.0,1.0,0.0,1080.0,1
1,01001020200,1.0,2055.0,723.0,0.0,142.0,6.91,471.0,316.0,842.0,...,9.5,49742.306258,1148.0,28.0,143.0,1.0,0.0,0.0,167.0,0
2,01001020300,1.0,3216.0,1291.0,0.0,19.0,0.59,731.0,598.0,2244.0,...,7.1,67187.346684,2350.0,84.0,111.0,1.0,1.0,0.0,612.0,1
3,01001020400,1.0,4246.0,1707.0,0.0,0.0,0.00,902.0,961.0,3578.0,...,15.3,64749.350412,2580.0,114.0,78.0,1.0,0.0,0.0,14.0,0
4,01001020501,1.0,4322.0,1778.0,0.0,12.0,0.28,953.0,770.0,3241.0,...,3.5,86373.699013,4114.0,9.0,53.0,1.0,0.0,0.0,102.0,0


In [5]:
df_2025.isna().sum()

,0
tract_id,0
clean__Urban,0
clean__population_total,0
clean__occupied_housing_units_total,0
clean__GroupQuartersFlag,0
clean__NUMGQTRS,0
clean__PCTGQTRS,0
clean__TractKids,0
clean__TractSeniors,0
clean__TractWhite,0


In [6]:
raw_test = pipe.predict(df_2025[model_cols])

output_test_results_df = df_2025.copy()
output_test_results_df["cluster"] = pd.Series(raw_test, index=df_2025.index).map(bundle["risk_map"])
output_test_results_df["risk"] = output_test_results_df["cluster"].map(bundle["risk_names"])

# adding state in second row of output file, first 2 digits of the tract_id
output_test_results_df["state"] = output_test_results_df["tract_id"].str[:2]
col = output_test_results_df.pop("state")
output_test_results_df.insert(1, col.name, col)

output_test_results_df.head()


,tract_id,state,clean__Urban,clean__population_total,clean__occupied_housing_units_total,clean__GroupQuartersFlag,clean__NUMGQTRS,clean__PCTGQTRS,clean__TractKids,clean__TractSeniors,...,clean__TractLOWI,clean__TractHUNV,clean__TractSNAP,clean__LATracts_half,clean__LATracts1,clean__LATractsVehicle_20,clean__low_access_population_1_20,FD_2,cluster,risk
0,01001020100,01,1.0,1775.0,646.0,0.0,0.0,0.00,405.0,284.0,...,1316.0,35.0,100.0,1.0,1.0,0.0,1080.0,1,0,low
1,01001020200,01,1.0,2055.0,723.0,0.0,142.0,6.91,471.0,316.0,...,1148.0,28.0,143.0,1.0,0.0,0.0,167.0,0,0,low
2,01001020300,01,1.0,3216.0,1291.0,0.0,19.0,0.59,731.0,598.0,...,2350.0,84.0,111.0,1.0,1.0,0.0,612.0,1,0,low
3,01001020400,01,1.0,4246.0,1707.0,0.0,0.0,0.00,902.0,961.0,...,2580.0,114.0,78.0,1.0,0.0,0.0,14.0,0,0,low
4,01001020501,01,1.0,4322.0,1778.0,0.0,12.0,0.28,953.0,770.0,...,4114.0,9.0,53.0,1.0,0.0,0.0,102.0,0,0,low


In [7]:
print(pd.crosstab(output_test_results_df["risk"], output_test_results_df["FD_2"], normalize="index"))
print(output_test_results_df.groupby("risk")["FD_2"].mean())

FD_2         0         1
risk                    
high  0.725746  0.274254
low   0.860576  0.139424
mid   0.950327  0.049673
risk
high    0.274254
low     0.139424
mid     0.049673
Name: FD_2, dtype: float64


In [8]:
output_test_results_df.head()

,tract_id,state,clean__Urban,clean__population_total,clean__occupied_housing_units_total,clean__GroupQuartersFlag,clean__NUMGQTRS,clean__PCTGQTRS,clean__TractKids,clean__TractSeniors,...,clean__TractLOWI,clean__TractHUNV,clean__TractSNAP,clean__LATracts_half,clean__LATracts1,clean__LATractsVehicle_20,clean__low_access_population_1_20,FD_2,cluster,risk
0,01001020100,01,1.0,1775.0,646.0,0.0,0.0,0.00,405.0,284.0,...,1316.0,35.0,100.0,1.0,1.0,0.0,1080.0,1,0,low
1,01001020200,01,1.0,2055.0,723.0,0.0,142.0,6.91,471.0,316.0,...,1148.0,28.0,143.0,1.0,0.0,0.0,167.0,0,0,low
2,01001020300,01,1.0,3216.0,1291.0,0.0,19.0,0.59,731.0,598.0,...,2350.0,84.0,111.0,1.0,1.0,0.0,612.0,1,0,low
3,01001020400,01,1.0,4246.0,1707.0,0.0,0.0,0.00,902.0,961.0,...,2580.0,114.0,78.0,1.0,0.0,0.0,14.0,0,0,low
4,01001020501,01,1.0,4322.0,1778.0,0.0,12.0,0.28,953.0,770.0,...,4114.0,9.0,53.0,1.0,0.0,0.0,102.0,0,0,low


In [10]:
#save result

output_test_results_df.to_csv("/content/drive/MyDrive/Food_Desert_Data/model_output_2025.csv", index=False)